In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from category_encoders import TargetEncoder

from sklearn.neighbors import KNeighborsRegressor

from sklearn.metrics import r2_score,mean_absolute_error, root_mean_squared_error

In [2]:
df = pd.read_csv('C:\\Users\\Admin\\OneDrive\\Desktop\\data science\\ML\\SUPERVISED LEARNING\\Dataset\\CAR DETAILS FROM CAR DEKHO.csv')
df

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner
0,Maruti 800 AC,2007,60000,70000,Petrol,Individual,Manual,First Owner
1,Maruti Wagon R LXI Minor,2007,135000,50000,Petrol,Individual,Manual,First Owner
2,Hyundai Verna 1.6 SX,2012,600000,100000,Diesel,Individual,Manual,First Owner
3,Datsun RediGO T Option,2017,250000,46000,Petrol,Individual,Manual,First Owner
4,Honda Amaze VX i-DTEC,2014,450000,141000,Diesel,Individual,Manual,Second Owner
...,...,...,...,...,...,...,...,...
4335,Hyundai i20 Magna 1.4 CRDi (Diesel),2014,409999,80000,Diesel,Individual,Manual,Second Owner
4336,Hyundai i20 Magna 1.4 CRDi,2014,409999,80000,Diesel,Individual,Manual,Second Owner
4337,Maruti 800 AC BSIII,2009,110000,83000,Petrol,Individual,Manual,Second Owner
4338,Hyundai Creta 1.6 CRDi SX Option,2016,865000,90000,Diesel,Individual,Manual,First Owner


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4340 entries, 0 to 4339
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   name           4340 non-null   str  
 1   year           4340 non-null   int64
 2   selling_price  4340 non-null   int64
 3   km_driven      4340 non-null   int64
 4   fuel           4340 non-null   str  
 5   seller_type    4340 non-null   str  
 6   transmission   4340 non-null   str  
 7   owner          4340 non-null   str  
dtypes: int64(3), str(5)
memory usage: 271.4 KB


In [4]:
df.corr(numeric_only=True)

,year,selling_price,km_driven
year,1.000000,0.413922,-0.419688
selling_price,0.413922,1.000000,-0.192289
km_driven,-0.419688,-0.192289,1.000000


In [5]:
df.nunique()

name             1491
year               27
selling_price     445
km_driven         770
fuel                5
seller_type         3
transmission        2
owner               5
dtype: int64

In [6]:
X = df.drop(columns='selling_price')
y = df.selling_price

In [7]:
xtrain, xtest, ytrain, ytest = train_test_split(X,y,train_size=0.8,random_state=42)

In [8]:
num_cols = X.select_dtypes(include='number').columns
obj_cols = X.select_dtypes(include='object')
obj_cols.columns

C:\Users\Admin\AppData\Local\Temp\ipykernel_9188\2735313497.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols = X.select_dtypes(include='object')


Index(['name', 'fuel', 'seller_type', 'transmission', 'owner'], dtype='str')

In [9]:
preprocessing = ColumnTransformer(
    transformers=[
        ('encode',OneHotEncoder(),['fuel', 'seller_type', 'transmission', 'owner']),
        ('encoder',TargetEncoder(),['name'])
    ],remainder='passthrough'
)

main_pipe = Pipeline(
    steps=[
        ('pre',preprocessing),
        ('model',KNeighborsRegressor())
    ]
)

In [10]:
import numpy as np
np.sqrt(4340)

np.float64(65.87867636800242)

In [11]:
grid_cv = GridSearchCV(
    estimator=main_pipe,
    param_grid={'model__n_neighbors':[2,6,10,25,50,60,64,66,70],
                'model__metric':['manhattan','euclidean']
                },
    cv=5,
    n_jobs=-1,
    verbose=2
)

In [ ]:
grid_cv.fit(xtrain,ytrain)

Fitting 5 folds for each of 18 candidates, totalling 90 fits


In [ ]:
grid_cv.best_params_

{'model__metric': 'manhattan', 'model__n_neighbors': 2}

In [ ]:
ytrain_pred = grid_cv.predict(xtrain)

In [ ]:
r2_score(ytrain,ytrain_pred)

0.9814099188113696

In [ ]:
ytest_pred = grid_cv.predict(xtest)

In [ ]:
r2_score(ytest,ytest_pred)

0.47293215960058443